In [36]:
#This is the notebook where LP is using python + Claude to analyse scraped data from AHCA's website. 
#The data source is the FEC Schedule A itemized receipts data via https://www.fec.gov/data/receipts/?committee_id=C00936559&two_year_transaction_period=2026&data_type=processed

In [37]:
import pandas as pd

In [38]:
## Now let's make a copy of our lastest CSV file that is ONLY individual contributors.

In [39]:
dfVindmanInd = pd.read_csv("LP_ind_Vindman_schedule_a-2026-08-18.csv")

In [40]:
dfVindmanInd.info()

<class 'pandas.DataFrame'>
RangeIndex: 17695 entries, 0 to 17694
Data columns (total 78 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   committee_id                           17695 non-null  str    
 1   committee_name                         17695 non-null  str    
 2   report_year                            17695 non-null  int64  
 3   report_type                            17695 non-null  str    
 4   image_number                           17695 non-null  int64  
 5   filing_form                            17695 non-null  str    
 6   link_id                                17695 non-null  int64  
 7   line_number                            17695 non-null  str    
 8   transaction_id                         17695 non-null  int64  
 9   file_number                            17695 non-null  int64  
 10  entity_type                            17695 non-null  str    
 11  entity_type_d

In [41]:
###Claude Prompt: Write simple code block by block (to copy and paste) to tally all contributions, 
###tally all Florida contributions, and then show the percent if Florida contributions compared to other states.

In [42]:
##IMPORTANT NOTE FROM LP. This code tallies the total contributions from individuals only. It does NOT include PAC donations.
##And it adds together both negative values like refunds and positive values. 

In [43]:
## How much did this candidate recieve in total from individuals (not PACS)?

In [44]:
total_contributions = dfVindmanInd['contribution_receipt_amount'].sum()
print(f"Total contributions: ${total_contributions:,.2f}")

Total contributions: $5,530,161.26


In [45]:
## How much did this candidate recieve in total from individuals (not PACS) in Florida?

In [46]:
florida_dfVindmanInd = dfVindmanInd[dfVindmanInd['contributor_state'] == 'FL']
florida_totalInd = florida_dfVindmanInd['contribution_receipt_amount'].sum()
print(f"Florida contributions: ${florida_totalInd:,.2f}")

Florida contributions: $1,851,271.15


In [47]:
## How much did this candidate recieve as a percentage from individuals (not PACS) in Florida versus other states?

In [48]:
other_states_total = total_contributions - florida_totalInd

florida_pct = (florida_totalInd / total_contributions) * 100
other_pct = (other_states_total / total_contributions) * 100

print(f"Florida: {florida_pct:.1f}% (${florida_totalInd:,.2f})")
print(f"Other states: {other_pct:.1f}% (${other_states_total:,.2f})")

Florida: 33.5% ($1,851,271.15)
Other states: 66.5% ($3,678,890.11)


In [49]:
## How much did this candidate recieve in total from individuals (not PACS) in all states?

In [50]:
state_totals = dfVindmanInd.groupby('contributor_state')['contribution_receipt_amount'].sum().sort_values(ascending=False)
print(state_totals)

contributor_state
FL    1851271.15
CA     787143.75
NY     559177.67
MA     358310.25
VA     190813.50
WA     162933.55
MD     117317.00
NJ     113997.50
CT     112725.00
TX     107847.50
IL      96896.76
DC      83057.00
PA      82269.88
CO      74391.00
MN      62297.00
NC      57314.00
GA      51050.00
OR      49073.50
AZ      44116.00
OH      41934.50
MI      41333.00
ME      39552.00
WI      36833.00
TN      34660.58
MO      32813.50
IN      28489.00
KY      22875.00
SC      21090.50
NH      20083.00
UT      19187.00
LA      18335.00
NV      17758.17
RI      17379.00
NM      16179.00
ZZ      15750.00
KS      15086.00
VT      15052.50
ID      13913.00
WY      13100.00
AL      12947.50
DE      11502.00
OK      11500.00
IA      10332.00
HI       8568.00
AR       8175.00
AK       7321.00
SD       4685.00
MT       4560.00
WV       2315.00
MS       1650.00
NE       1600.00
ND        625.00
VI        500.00
AE        475.00
Name: contribution_receipt_amount, dtype: float64


In [51]:
## How much did this candidate recieve in total from individuals (not PACS) in all states. 
## How many individual TRANSACTIONS were there? (num_contributions)

In [52]:
##IMPORTANT NOTE: This counts individual contribution lines and not individual donors.

In [53]:
state_summary = dfVindmanInd.groupby('contributor_state')['contribution_receipt_amount'].agg(['sum', 'count']).sort_values('sum', ascending=False)
state_summary.columns = ['total_amount', 'num_contributions']
print(state_summary)

                   total_amount  num_contributions
contributor_state                                 
FL                   1851271.15               5745
CA                    787143.75               2246
NY                    559177.67               1112
MA                    358310.25                750
VA                    190813.50                627
WA                    162933.55                591
MD                    117317.00                404
NJ                    113997.50                369
CT                    112725.00                280
TX                    107847.50                571
IL                     96896.76                445
DC                     83057.00                128
PA                     82269.88                368
CO                     74391.00                288
MN                     62297.00                222
NC                     57314.00                317
GA                     51050.00                172
OR                     49073.50

In [54]:
###Claude prompt: Show each state as a percentage of total

In [55]:
## The total contributions from each state as a percentage. 

In [56]:
total_contributions = dfVindmanInd['contribution_receipt_amount'].sum()

state_summary = dfVindmanInd.groupby('contributor_state')['contribution_receipt_amount'].sum().sort_values(ascending=False)
state_summary_pct = (state_summary / total_contributions * 100).round(1)

for state, amount in state_summary.items():
    print(f"{state}: ${amount:,.2f} ({state_summary_pct[state]}%)")

FL: $1,851,271.15 (33.5%)
CA: $787,143.75 (14.2%)
NY: $559,177.67 (10.1%)
MA: $358,310.25 (6.5%)
VA: $190,813.50 (3.5%)
WA: $162,933.55 (2.9%)
MD: $117,317.00 (2.1%)
NJ: $113,997.50 (2.1%)
CT: $112,725.00 (2.0%)
TX: $107,847.50 (2.0%)
IL: $96,896.76 (1.8%)
DC: $83,057.00 (1.5%)
PA: $82,269.88 (1.5%)
CO: $74,391.00 (1.3%)
MN: $62,297.00 (1.1%)
NC: $57,314.00 (1.0%)
GA: $51,050.00 (0.9%)
OR: $49,073.50 (0.9%)
AZ: $44,116.00 (0.8%)
OH: $41,934.50 (0.8%)
MI: $41,333.00 (0.7%)
ME: $39,552.00 (0.7%)
WI: $36,833.00 (0.7%)
TN: $34,660.58 (0.6%)
MO: $32,813.50 (0.6%)
IN: $28,489.00 (0.5%)
KY: $22,875.00 (0.4%)
SC: $21,090.50 (0.4%)
NH: $20,083.00 (0.4%)
UT: $19,187.00 (0.3%)
LA: $18,335.00 (0.3%)
NV: $17,758.17 (0.3%)
RI: $17,379.00 (0.3%)
NM: $16,179.00 (0.3%)
ZZ: $15,750.00 (0.3%)
KS: $15,086.00 (0.3%)
VT: $15,052.50 (0.3%)
ID: $13,913.00 (0.3%)
WY: $13,100.00 (0.2%)
AL: $12,947.50 (0.2%)
DE: $11,502.00 (0.2%)
OK: $11,500.00 (0.2%)
IA: $10,332.00 (0.2%)
HI: $8,568.00 (0.2%)
AR: $8,175.00 (0.1